In [1]:
import sys
from pathlib import Path
import json

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import *
from src.data_split import *
from src.metrics import *
from src.modeling import *

import warnings
warnings.filterwarnings("ignore")

In [2]:
with open("../data/features/features.json", "r", encoding="utf-8") as f:
    selected_features_300 = json.load(f)
selected_features_250 = selected_features_300[:250]
selected_features_200 = selected_features_300[:200]
selected_features_150 = selected_features_300[:150]
selected_features_100 = selected_features_300[:100]
all_feature_sets = [selected_features_300, selected_features_250, selected_features_200, selected_features_150, selected_features_100]

In [3]:
train_df = pd.read_csv('../data/train_test/holdout/train_full.csv', index_col='id', parse_dates=['timestamp'])

time_folds = get_time_folds()

In [4]:
model = CatBoostRegressor(
iterations=3000,
learning_rate=0.05,
depth=6,
loss_function="RMSE",
eval_metric="RMSE",
random_seed=42,
early_stopping_rounds=200,
verbose=200,
)

all_cat_results = []

for i, feature_set in enumerate(all_feature_sets, start=0):

    feature_set_name = f"feature_set_{i}_{len(feature_set)}"

    cat_quality, oof_pred_cat, cat_models, features_import_cat, cat_features = run_boosting_cv(
        train_df=train_df,
        time_folds=time_folds,
        split_fold=split_fold,
        model_factory=model,
        model_name=f"catboost_{feature_set_name}",
        booster="catboost",
        preprocessing_func=prep_with_features_eng,
        selected_features=feature_set,
    )

    all_cat_results.append(cat_quality)

all_cat_results = pd.concat(all_cat_results, ignore_index=True)
all_cat_results

0:	learn: 0.6059160	test: 0.5831971	best: 0.5831971 (0)	total: 217ms	remaining: 10m 49s
200:	learn: 0.4617106	test: 0.4665426	best: 0.4664550 (198)	total: 14.6s	remaining: 3m 22s
400:	learn: 0.4199752	test: 0.4664940	best: 0.4660587 (280)	total: 29.7s	remaining: 3m 12s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4660586924
bestIteration = 280

Shrink model to first 281 iterations.
0:	learn: 0.5964904	test: 0.6022952	best: 0.6022952 (0)	total: 82.7ms	remaining: 4m 8s
200:	learn: 0.4618201	test: 0.4794820	best: 0.4794706 (199)	total: 15.3s	remaining: 3m 33s
400:	learn: 0.4318915	test: 0.4757015	best: 0.4757015 (400)	total: 29.6s	remaining: 3m 11s
600:	learn: 0.4064592	test: 0.4764056	best: 0.4757015 (400)	total: 43.9s	remaining: 2m 55s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4757015305
bestIteration = 400

Shrink model to first 401 iterations.
0:	learn: 0.5972664	test: 0.5995705	best: 0.5995705 (0)	total: 79.4ms	remaining: 3m 58s
200:

,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,catboost_feature_set_0_300,0.466059,0.475702,0.439242,0.460334,0.018892
1,catboost_feature_set_1_250,0.465576,0.477364,0.437078,0.460006,0.020712
2,catboost_feature_set_2_200,0.466029,0.476668,0.434861,0.459186,0.021727
3,catboost_feature_set_3_150,0.465633,0.474431,0.435585,0.458550,0.020368
4,catboost_feature_set_4_100,0.464042,0.474308,0.436642,0.458331,0.019472


In [5]:
model = LGBMRegressor(
n_estimators=3000,
learning_rate=0.05,
num_leaves=31,
max_depth=-1,
min_child_samples=20,
subsample=0.8,
colsample_bytree=0.8,
objective="regression",
random_state=42,
n_jobs=-1
)

all_lgb_results = []

for i, feature_set in enumerate(all_feature_sets, start=0):

    feature_set_name = f"feature_set_{i}_{len(feature_set)}"

    lightgbm_quality, oof_pred_lightgbm, lightgbm_models, features_import_light, lgb_features = run_boosting_cv(
        train_df=train_df,
        time_folds=time_folds,
        split_fold=split_fold,
        model_factory=model,
        model_name=f"lightgbm_{feature_set_name}",
        booster="lightgbm",
        preprocessing_func=prep_with_features_eng,
        selected_features=feature_set,
    )

    all_lgb_results.append(lightgbm_quality)

all_lgb_results = pd.concat(all_lgb_results, ignore_index=True)
all_lgb_results

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022301 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41933
[LightGBM] [Info] Number of data points in the train set: 10160, number of used features: 300
[LightGBM] [Info] Start training from score 15.501434
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.478675	valid_0's l2: 0.22913
Early stopping, best iteration is:
[88]	valid_0's rmse: 0.475619	valid_0's l2: 0.226214
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024962 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43061
[LightGBM] [Info] Number of data points in the train set: 15231, number of used features: 300
[LightGBM] [Info] Start training from score 15.535907
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.484554	valid_0's l2: 

,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,lightgbm_feature_set_0_300,0.475619,0.483207,0.444582,0.467803,0.020465
1,lightgbm_feature_set_1_250,0.475364,0.481824,0.445756,0.467648,0.019232
2,lightgbm_feature_set_2_200,0.475977,0.482522,0.445855,0.468118,0.019556
3,lightgbm_feature_set_3_150,0.474275,0.483879,0.444778,0.467644,0.020376
4,lightgbm_feature_set_4_100,0.475325,0.480109,0.445949,0.467128,0.018497


In [6]:
model = XGBRegressor(
n_estimators=3000,
learning_rate=0.05,
max_depth=6,
min_child_weight=1,
subsample=0.8,
colsample_bytree=0.8,
objective="reg:squarederror",
eval_metric="rmse",
tree_method="hist",
enable_categorical=True,
early_stopping_rounds=200,
random_state=42,
n_jobs=-1,
)

all_xgb_results = []

for i, feature_set in enumerate(all_feature_sets, start=0):

    feature_set_name = f"feature_set_{i}_{len(feature_set)}"

    xgboost_quality, oof_pred_xgb, xgb_models, features_import_xgb, xgb_features = run_boosting_cv(
        train_df=train_df,
        time_folds=time_folds,
        split_fold=split_fold,
        model_factory=model,
        model_name=f"xgboost_{feature_set_name}",
        booster="xgboost",
        preprocessing_func=prep_with_features_eng,
        selected_features=feature_set,
    )

    all_xgb_results.append(xgboost_quality)

all_xgb_results = pd.concat(all_xgb_results, ignore_index=True)
all_xgb_results

[0]	validation_0-rmse:0.58076
[200]	validation_0-rmse:0.49087
[269]	validation_0-rmse:0.49615
[0]	validation_0-rmse:0.60085
[200]	validation_0-rmse:0.49832
[276]	validation_0-rmse:0.50423
[0]	validation_0-rmse:0.59782
[200]	validation_0-rmse:0.45794
[319]	validation_0-rmse:0.47052
[0]	validation_0-rmse:0.58064
[200]	validation_0-rmse:0.48925
[269]	validation_0-rmse:0.49387
[0]	validation_0-rmse:0.60070
[200]	validation_0-rmse:0.49108
[289]	validation_0-rmse:0.49870
[0]	validation_0-rmse:0.59735
[200]	validation_0-rmse:0.46182
[304]	validation_0-rmse:0.47245
[0]	validation_0-rmse:0.58098
[200]	validation_0-rmse:0.48776
[269]	validation_0-rmse:0.49229
[0]	validation_0-rmse:0.60068
[200]	validation_0-rmse:0.49155
[287]	validation_0-rmse:0.49909
[0]	validation_0-rmse:0.59780
[200]	validation_0-rmse:0.45972
[318]	validation_0-rmse:0.47308
[0]	validation_0-rmse:0.58062
[200]	validation_0-rmse:0.48987
[257]	validation_0-rmse:0.49387
[0]	validation_0-rmse:0.60082
[200]	validation_0-rmse:0.4924

,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
0,xgboost_feature_set_0_300,0.481879,0.490196,0.450918,0.474331,0.020698
1,xgboost_feature_set_1_250,0.481422,0.485829,0.452636,0.473296,0.018027
2,xgboost_feature_set_2_200,0.480081,0.486958,0.452052,0.473030,0.018490
3,xgboost_feature_set_3_150,0.480055,0.486229,0.452197,0.472827,0.018130
4,xgboost_feature_set_4_100,0.480532,0.480349,0.448662,0.469848,0.018348


In [32]:
journal_quality = pd.read_csv('../data/journals/models_quality.csv', index_col=0).drop(['Unnamed: 0.1', 'Unnamed: 0'], axis=1)
journal_quality = pd.concat([journal_quality, all_cat_results, all_lgb_results, all_xgb_results])
journal_quality = journal_quality.sort_values('mean_rmsle').drop_duplicates('model_name')
journal_quality

,model_name,fold_1,fold_2,fold_3,mean_rmsle,rmsle_std
10,catboost_feature_set_4_100,0.464042,0.474308,0.436642,0.458331,0.019472
3,catboost_feature_set_3_150,0.465633,0.474431,0.435585,0.458550,0.020368
8,catboost_feature_set_2_200,0.466029,0.476668,0.434861,0.459186,0.021727
3,catboost_eng,0.466676,0.475195,0.436673,0.459514,0.020235
1,catboost_feature_set_1_250,0.465576,0.477364,0.437078,0.460006,0.020712
6,catboost_feature_set_0_300,0.466059,0.475702,0.439242,0.460334,0.018892
0,catboost_baseline,0.469224,0.477262,0.439019,0.461835,0.020164
4,lightgbm_feature_set_4_100,0.475325,0.480109,0.445949,0.467128,0.018497
4,lightgbm_eng,0.475595,0.482197,0.444557,0.467450,0.020099
14,lightgbm_feature_set_3_150,0.474275,0.483879,0.444778,0.467644,0.020376


In [33]:
journal_quality.to_csv('../data/journals/models_quality.csv')

catboost_feature_set_4_100
catboost_feature_set_2_200
lightgbm_feature_set_4_100
lightgbm_eng
xgboost_feature_set_4_100

In [9]:
model = CatBoostRegressor(
iterations=3000,
learning_rate=0.05,
depth=6,
loss_function="RMSE",
eval_metric="RMSE",
random_seed=42,
early_stopping_rounds=200,
verbose=200,
)

cat_quality_100, oof_pred_cat_100, cat_models_100, features_import_cat_100, cat_features_100 = run_boosting_cv(
train_df=train_df,
time_folds=time_folds,
split_fold=split_fold,
model_factory=model,
model_name=f"catboost_100",
booster="catboost",
preprocessing_func=prep_with_features_eng,
selected_features=selected_features_100,
)

0:	learn: 0.6055542	test: 0.5822875	best: 0.5822875 (0)	total: 45.8ms	remaining: 2m 17s
200:	learn: 0.4614664	test: 0.4655133	best: 0.4655133 (200)	total: 9.62s	remaining: 2m 13s
400:	learn: 0.4213783	test: 0.4656060	best: 0.4645219 (271)	total: 18.8s	remaining: 2m 1s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4645218548
bestIteration = 271

Shrink model to first 272 iterations.
0:	learn: 0.5965716	test: 0.6020475	best: 0.6020475 (0)	total: 78.3ms	remaining: 3m 54s
200:	learn: 0.4616161	test: 0.4772653	best: 0.4772482 (198)	total: 9.87s	remaining: 2m 17s
400:	learn: 0.4337811	test: 0.4748058	best: 0.4747821 (397)	total: 19.8s	remaining: 2m 8s
600:	learn: 0.4074007	test: 0.4761126	best: 0.4746052 (436)	total: 30.2s	remaining: 2m
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4746052054
bestIteration = 436

Shrink model to first 437 iterations.
0:	learn: 0.5969989	test: 0.5985636	best: 0.5985636 (0)	total: 73.4ms	remaining: 3m 40s
200:	lear

In [10]:
model = CatBoostRegressor(
iterations=3000,
learning_rate=0.05,
depth=6,
loss_function="RMSE",
eval_metric="RMSE",
random_seed=42,
early_stopping_rounds=200,
verbose=200,
)

cat_quality_200, oof_pred_cat_200, cat_models_200, features_import_cat_200, cat_features_200 = run_boosting_cv(
train_df=train_df,
time_folds=time_folds,
split_fold=split_fold,
model_factory=model,
model_name=f"catboost_200",
booster="catboost",
preprocessing_func=prep_with_features_eng,
selected_features=selected_features_200,
)

0:	learn: 0.6049086	test: 0.5817492	best: 0.5817492 (0)	total: 116ms	remaining: 5m 46s
200:	learn: 0.4612444	test: 0.4671220	best: 0.4670315 (181)	total: 11.5s	remaining: 2m 39s
400:	learn: 0.4196699	test: 0.4672210	best: 0.4665187 (293)	total: 23.2s	remaining: 2m 30s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4665186678
bestIteration = 293

Shrink model to first 294 iterations.
0:	learn: 0.5964465	test: 0.6020193	best: 0.6020193 (0)	total: 91.4ms	remaining: 4m 33s
200:	learn: 0.4610970	test: 0.4794653	best: 0.4794653 (200)	total: 12.1s	remaining: 2m 49s
400:	learn: 0.4312324	test: 0.4778211	best: 0.4776335 (381)	total: 23.5s	remaining: 2m 32s
600:	learn: 0.4040197	test: 0.4779929	best: 0.4766525 (521)	total: 34.8s	remaining: 2m 18s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.4766525052
bestIteration = 521

Shrink model to first 522 iterations.
0:	learn: 0.5971461	test: 0.5994030	best: 0.5994030 (0)	total: 78.6ms	remaining: 3m 55s
200:

In [11]:
model = LGBMRegressor(
n_estimators=3000,
learning_rate=0.05,
num_leaves=31,
max_depth=-1,
min_child_samples=20,
subsample=0.8,
colsample_bytree=0.8,
objective="regression",
random_state=42,
n_jobs=-1
)

lightgbm_quality_100, oof_pred_lightgbm_100, lightgbm_models_100, features_import_light_100, lgb_features_100 = run_boosting_cv(
train_df=train_df,
time_folds=time_folds,
split_fold=split_fold,
model_factory=model,
model_name=f"lightgbm_100",
booster="lightgbm",
preprocessing_func=prep_with_features_eng,
selected_features=selected_features_100,
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007055 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19512
[LightGBM] [Info] Number of data points in the train set: 10160, number of used features: 100
[LightGBM] [Info] Start training from score 15.501434
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.477482	valid_0's l2: 0.227989
Early stopping, best iteration is:
[117]	valid_0's rmse: 0.475325	valid_0's l2: 0.225934
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008842 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19751
[LightGBM] [Info] Number of data points in the train set: 15231, number of used features: 100
[LightGBM] [Info] Start training from score 15.535907
Training until validation scores don't improve for 200 rounds
[200]	valid_0's rmse: 0.481467	valid_0's l2

In [12]:
model = XGBRegressor(
n_estimators=3000,
learning_rate=0.05,
max_depth=6,
min_child_weight=1,
subsample=0.8,
colsample_bytree=0.8,
objective="reg:squarederror",
eval_metric="rmse",
tree_method="hist",
enable_categorical=True,
early_stopping_rounds=200,
random_state=42,
n_jobs=-1,
)

xgboost_quality_100, oof_pred_xgb_100, xgb_models_100, features_import_xgb_100, xgb_features_100 = run_boosting_cv(
train_df=train_df,
time_folds=time_folds,
split_fold=split_fold,
model_factory=model,
model_name=f"xgboost_100",
booster="xgboost",
preprocessing_func=prep_with_features_eng,
selected_features=selected_features_100,
)

[0]	validation_0-rmse:0.58082
[200]	validation_0-rmse:0.49009
[258]	validation_0-rmse:0.49349
[0]	validation_0-rmse:0.60099
[200]	validation_0-rmse:0.48209
[335]	validation_0-rmse:0.48875
[0]	validation_0-rmse:0.59738
[200]	validation_0-rmse:0.45363
[303]	validation_0-rmse:0.46162


In [13]:
oof_pred_xgb_100 = oof_pred_xgb_100.reset_index(drop=True)
oof_pred_lightgbm_100 = oof_pred_lightgbm_100.reset_index(drop=True)
oof_pred_cat_100 = oof_pred_cat_100.reset_index(drop=True)
oof_pred_cat_200 = oof_pred_cat_200.reset_index(drop=True)

pred_journal = pd.read_csv('../data/journals/models_preds.csv').drop('Unnamed: 0', axis=1)
pred_journal = pred_journal.merge(
        oof_pred_xgb_100[["id", "fold", "val_block", "xgboost_100_pred", "xgboost_100_error"]],
        on=["id", "fold", "val_block"],
        how="left"
        ).merge(
        oof_pred_lightgbm_100[["id", "fold", "val_block", "lightgbm_100_pred", "lightgbm_100_error"]],
        on=["id", "fold", "val_block"],
        how="left"
        ).merge(
        oof_pred_cat_100[["id", "fold", "val_block", "catboost_100_pred", "catboost_100_error"]],
        on=["id", "fold", "val_block"],
        how="left"
        ).merge(
        oof_pred_cat_200[["id", "fold", "val_block", "catboost_200_pred", "catboost_200_error"]],
        on=["id", "fold", "val_block"],
        how="left"
        )

In [14]:
pred_journal

,id,fold,val_block,y_true,catboost_baseline_pred,catboost_baseline_error,xgboost_baseline_pred,xgboost_baseline_error,lightgbm_baseline_pred,lightgbm_baseline_error,...,catboost_eng_pred,catboost_eng_error,xgboost_100_pred,xgboost_100_error,lightgbm_100_pred,lightgbm_100_error,catboost_100_pred,catboost_100_error,catboost_200_pred,catboost_200_error
0,10165,1,B3,8552548.0,8.038536e+06,-5.140120e+05,7798804.0,-753744.00,8.437683e+06,-1.148646e+05,...,7.814576e+06,-7.379722e+05,8057891.00,-494657.00,8.586025e+06,3.347748e+04,7.779673e+06,-7.728751e+05,7.842445e+06,-7.101031e+05
1,10166,1,B3,11400000.0,8.497058e+06,-2.902942e+06,7329757.5,-4070242.50,7.929031e+06,-3.470969e+06,...,8.703967e+06,-2.696033e+06,7693272.00,-3706728.00,8.322124e+06,-3.077876e+06,8.562741e+06,-2.837259e+06,8.465925e+06,-2.934075e+06
2,10167,1,B3,14021000.0,1.797495e+07,3.953946e+06,17850364.0,3829364.00,1.848124e+07,4.460243e+06,...,1.780653e+07,3.785527e+06,17437228.00,3416228.00,1.995632e+07,5.935324e+06,1.712896e+07,3.107960e+06,1.827596e+07,4.254959e+06
3,10168,1,B3,5395417.0,5.262100e+06,-1.333166e+05,5201082.0,-194335.00,5.211459e+06,-1.839583e+05,...,5.224533e+06,-1.708841e+05,5223286.00,-172131.00,5.326887e+06,-6.853047e+04,5.235497e+06,-1.599201e+05,5.165120e+06,-2.302970e+05
4,10169,1,B3,4462000.0,4.163969e+06,-2.980311e+05,4109457.0,-352543.00,3.971648e+06,-4.903524e+05,...,4.202900e+06,-2.591002e+05,4095310.00,-366690.00,4.019895e+06,-4.421050e+05,4.129915e+06,-3.320846e+05,4.068272e+06,-3.937275e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15273,25440,3,B5,4082035.0,4.229707e+06,1.476720e+05,4000694.0,-81341.00,4.012889e+06,-6.914551e+04,...,4.237237e+06,1.552024e+05,3786192.00,-295843.00,4.043962e+06,-3.807309e+04,4.248231e+06,1.661961e+05,4.021849e+06,-6.018628e+04
15274,25441,3,B5,3827681.0,4.110387e+06,2.827063e+05,3962631.2,134950.25,3.946705e+06,1.190242e+05,...,4.069587e+06,2.419060e+05,3925832.50,98151.50,4.030422e+06,2.027407e+05,4.027528e+06,1.998470e+05,4.159432e+06,3.317513e+05
15275,25442,3,B5,7250000.0,5.455821e+06,-1.794179e+06,4715719.0,-2534281.00,5.714960e+06,-1.535040e+06,...,4.592519e+06,-2.657481e+06,5651981.00,-1598019.00,5.704240e+06,-1.545760e+06,5.198769e+06,-2.051231e+06,4.501789e+06,-2.748211e+06
15276,25443,3,B5,8550000.0,6.752915e+06,-1.797085e+06,7601352.0,-948648.00,7.871548e+06,-6.784524e+05,...,6.850931e+06,-1.699069e+06,7299949.50,-1250050.50,6.741996e+06,-1.808004e+06,6.970526e+06,-1.579474e+06,6.811173e+06,-1.738827e+06


In [15]:
pred_journal[['xgboost_100_error', 'catboost_100_error', 'catboost_200_error', 'lightgbm_100_error', 'lightgbm_eng_error']].corr()

,xgboost_100_error,catboost_100_error,catboost_200_error,lightgbm_100_error,lightgbm_eng_error
xgboost_100_error,1.000000,0.943625,0.946191,0.963399,0.961923
catboost_100_error,0.943625,1.000000,0.983880,0.962206,0.953481
catboost_200_error,0.946191,0.983880,1.000000,0.960631,0.957144
lightgbm_100_error,0.963399,0.962206,0.960631,1.000000,0.979956
lightgbm_eng_error,0.961923,0.953481,0.957144,0.979956,1.000000


catboost_100 xgboost_100 lightgbm_100

In [16]:
with open("../data/features/features_100.json", "w", encoding="utf-8") as f:
    json.dump(selected_features_100, f, ensure_ascii=False, indent=4)

In [17]:
pred_journal.to_csv('../data/journals/models_preds.csv')